# Extracción de keypoints con YOLO11-Pose

**Proyecto:** detección y clasificación de golpes de boxeo mediante **YOLO11-Pose + LSTM**.

## Objetivo

Procesar los videos de cada clase de golpe para detectar los puntos clave del cuerpo (*keypoints*) en cada fotograma muestreado. Se guardan los **17 keypoints COCO** para conservar información y, en la preparación del LSTM, se seleccionan inicialmente los 8 puntos de hombros, codos, muñecas y caderas.

**Clases esperadas** (una carpeta dentro de `videos/` por cada una): `jab`, `cross`, `hook`, `uppercut`, `livershot` (golpe al hígado) y `guardia`.

## Archivos generados

- `salida/keypoints/<clase>/<video>.npz`: los 17 keypoints por fotograma procesado, sus confianzas, detección y metadatos de muestreo.
- `salida/overlays/<clase>/<video>.mp4`: video de revisión con solo los 8 keypoints del modelo y sus conexiones; no dibuja nariz ni piernas.
- `salida/manifest.csv`: resumen y estado de procesamiento de cada video.
- `salida/balance_clases.png`: cantidad de videos y fotogramas por clase.
- `salida/muestras_esqueleto.png`: ejemplos de las detecciones obtenidas.

**Orden de ejecución:** ejecutar las celdas de arriba hacia abajo. Este notebook está preparado para Google Colab y utiliza una carpeta del proyecto ubicada en Google Drive.

## 1. Instalar e importar las bibliotecas

La siguiente celda instala Ultralytics, que permite utilizar YOLO11-Pose.

In [ ]:
!pip -q install ultralytics

In [ ]:
# Bibliotecas para archivos, videos, datos, gráficos y modelo
import shutil
import subprocess
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm
from ultralytics import YOLO

# Conectar Google Drive (entorno Google Colab)
from google.colab import drive

drive.mount('/content/drive')

# Carpeta principal del proyecto en Google Drive
BASE = Path('/content/drive/MyDrive/boxeo_proyecto')

print('GPU disponible:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print(
        'AVISO: no hay GPU. En Colab, ve a '
        'Entorno de ejecución > Cambiar tipo de entorno > T4 GPU.'
    )

## 2. Configurar el procesamiento

Aquí se concentran los parámetros ajustables:

- `REPROCESAR`: con `False` se reutilizan archivos compatibles; los archivos antiguos o con otros parámetros se reprocesan automáticamente.
- `FPS_OBJETIVO`: frecuencia temporal de muestreo. El muestreo permite obtener aproximadamente 20 fotogramas por segundo sin redondear el salto a un entero fijo.
- `PUNTOS_TORSO`: los 8 keypoints que usaremos inicialmente en el LSTM. En los archivos se guardan los 17 para permitir experimentos futuros.
- `UMBRAL_SIN_DETECCION` y `UMBRAL_CONF`: criterios para marcar videos que requieren revisión.

La primera ejecución con esta versión reprocesará los `.npz` que no tengan la nueva versión de formato o el FPS objetivo solicitado.

In [ ]:
# Modelo de detección de pose
MODELO = 'yolo11m-pose.pt'

# Extensiones de video que se procesarán
EXTENSIONES = {'.mp4', '.mov', '.m4v', '.avi'}

# Si es False, se omiten los videos que ya tienen keypoints guardados
# (solo se reutilizan los archivos con el formato actual; los antiguos se reprocesan automáticamente)
REPROCESAR = False

# Frecuencia objetivo de muestreo. Poner None para conservar todos los frames.
# Se permite un salto fraccionario (por ejemplo, 30 -> 20 fps) para reducir el error temporal.
FPS_OBJETIVO = 20
VERSION_FORMATO = 2

if FPS_OBJETIVO is not None and FPS_OBJETIVO <= 0:
    raise ValueError('FPS_OBJETIVO debe ser mayor que 0 o None.')

# Criterios para marcar un video como REVISAR
UMBRAL_SIN_DETECCION = 0.20  # Proporción máxima de frames sin persona detectada
UMBRAL_CONF = 0.50           # Confianza media mínima para los puntos seleccionados

# Tamaño máximo del lado más largo de los videos de revisión
LADO_MAX_OVERLAY = 1280

# Nombres de los 17 keypoints COCO, en el mismo orden en que se guardan en los archivos .npz
NOMBRES_COCO = [
    'nariz', 'ojo_izq', 'ojo_der', 'oreja_izq', 'oreja_der',
    'hombro_izq', 'hombro_der', 'codo_izq', 'codo_der',
    'muneca_izq', 'muneca_der', 'cadera_izq', 'cadera_der',
    'rodilla_izq', 'rodilla_der', 'tobillo_izq', 'tobillo_der',
]

# Keypoints que usará el modelo: hombros (5, 6), codos (7, 8), muñecas (9, 10) y caderas (11, 12)
PUNTOS_TORSO = [5, 6, 7, 8, 9, 10, 11, 12]
print('Keypoints del modelo:', [f'{i}:{NOMBRES_COCO[i]}' for i in PUNTOS_TORSO])

# Pares de keypoints que se unen con una línea en el video de revisión
CONEXIONES = [(5, 6), (5, 7), (7, 9), (6, 8), (8, 10), (5, 11), (6, 12), (11, 12)]

# Carpetas de entrada y salida
VIDEOS = BASE / 'videos'
SALIDA = BASE / 'salida'
KP_DIR = SALIDA / 'keypoints'
OV_DIR = SALIDA / 'overlays'

# Comprobar que exista la carpeta de videos
if not VIDEOS.exists():
    raise FileNotFoundError(
        f'No encuentro la carpeta {VIDEOS}. '
        'Revisa que exista en Drive con ese nombre exacto.'
    )

# Crear las carpetas de salida si todavía no existen
KP_DIR.mkdir(parents=True, exist_ok=True)
OV_DIR.mkdir(parents=True, exist_ok=True)

# Cada subcarpeta dentro de "videos" representa una clase de golpe
clases = sorted(carpeta.name for carpeta in VIDEOS.iterdir() if carpeta.is_dir())
print('Clases encontradas:', clases)


## 3. Cargar el modelo YOLO11-Pose

In [ ]:
model = YOLO(MODELO)
print('Modelo cargado:', MODELO)

## 4. Definir las funciones de procesamiento

Se utilizan tres funciones:

1. `a_h264`: intenta convertir el video de revisión al formato H.264.
2. `procesar_video`: analiza el video fotograma a fotograma (según `FPS_OBJETIVO`) y extrae los **17 keypoints** de la persona de mayor tamaño. El video de revisión dibuja solo los 8 keypoints del modelo y sus conexiones.
3. `estadisticas`: calcula métricas sobre los 8 keypoints del modelo y determina si el video queda como `OK` o `REVISAR`.

In [ ]:
def a_h264(origen: Path, destino: Path):
    """Convierte un video temporal a H.264; si falla, conserva el temporal."""
    try:
        subprocess.run(
            [
                'ffmpeg', '-y', '-loglevel', 'error',
                '-i', str(origen),
                '-vcodec', 'libx264',
                '-pix_fmt', 'yuv420p',
                str(destino),
            ],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
        )
        if origen.exists():
            origen.unlink()
    except Exception as error:
        # Si ffmpeg falla, retirar una posible salida parcial y conservar el video temporal.
        if destino.exists():
            destino.unlink()
        if origen.exists():
            shutil.move(str(origen), str(destino))
        print('  (aviso) No se pudo convertir a H.264; se conserva el video generado:', error)


def procesar_video(ruta_video: Path, ruta_overlay: Path):
    """Extrae 17 keypoints y crea un video de revisión con solo los 8 puntos elegidos.

    El muestreo se hace con posiciones de fotograma fraccionarias para aproximarse
    mejor a FPS_OBJETIVO (por ejemplo, 30 -> 20 fps). Devuelve keypoints,
    confianzas, detecciones, FPS efectivo, muestra, FPS original y salto medio.
    """
    cap = cv2.VideoCapture(str(ruta_video))
    if not cap.isOpened():
        raise RuntimeError(f'No se pudo abrir el video: {ruta_video}')

    fps_original = cap.get(cv2.CAP_PROP_FPS)
    if not np.isfinite(fps_original) or fps_original <= 0:
        fps_original = 30.0

    # Si se pide un FPS mayor al original, no inventamos fotogramas.
    fps = min(float(FPS_OBJETIVO), fps_original) if FPS_OBJETIVO else fps_original
    salto_medio = fps_original / fps

    ruta_temporal = ruta_overlay.with_name(ruta_overlay.stem + '_tmp.mp4')
    writer = None
    keypoints, confianzas, detecciones = [], [], []
    muestra = None
    ultimo_frame = None
    indice_frame = -1
    siguiente_frame_a_muestrear = 0.0

    while True:
        correcto, frame = cap.read()
        if not correcto:
            break

        indice_frame += 1
        # Selección temporal sin forzar el salto a un entero.
        if indice_frame + 1e-9 < siguiente_frame_a_muestrear:
            continue
        siguiente_frame_a_muestrear += salto_medio

        resultado = model(frame, verbose=False)[0]
        hay_persona = (
            resultado.keypoints is not None
            and resultado.boxes is not None
            and len(resultado.boxes) > 0
        )
        frame_anotado = frame.copy()

        if hay_persona:
            cajas = resultado.boxes.xywh
            areas = cajas[:, 2] * cajas[:, 3]
            indice_persona = int(areas.argmax())
            puntos_originales = resultado.keypoints.xy[indice_persona].cpu().numpy()

            if resultado.keypoints.conf is not None:
                confianza_original = resultado.keypoints.conf[indice_persona].cpu().numpy()
            else:
                confianza_original = np.ones(17, dtype=np.float32)

            # Guardamos los 17 puntos; PUNTOS_TORSO selecciona los 8 usados inicialmente por el LSTM.
            keypoints.append(puntos_originales.astype(np.float32))
            confianzas.append(confianza_original.astype(np.float32))
            detecciones.append(True)

            # Dibujar solo conexiones entre los 8 puntos seleccionados.
            for a, b in CONEXIONES:
                if (
                    confianza_original[a] >= 0.5
                    and confianza_original[b] >= 0.5
                    and (puntos_originales[a] > 0).all()
                    and (puntos_originales[b] > 0).all()
                ):
                    cv2.line(
                        frame_anotado,
                        (int(puntos_originales[a][0]), int(puntos_originales[a][1])),
                        (int(puntos_originales[b][0]), int(puntos_originales[b][1])),
                        (0, 200, 255), 3,
                    )

            for i in PUNTOS_TORSO:
                x, y = puntos_originales[i]
                if x > 0 and y > 0 and confianza_original[i] >= 0.5:
                    cv2.circle(frame_anotado, (int(x), int(y)), 6, (0, 255, 0), -1)
        else:
            keypoints.append(np.zeros((17, 2), dtype=np.float32))
            confianzas.append(np.zeros(17, dtype=np.float32))
            detecciones.append(False)

        lado_mayor = max(frame_anotado.shape[:2])
        if lado_mayor > LADO_MAX_OVERLAY:
            escala = LADO_MAX_OVERLAY / lado_mayor
            nuevo_tamano = (
                int(frame_anotado.shape[1] * escala),
                int(frame_anotado.shape[0] * escala),
            )
            frame_anotado = cv2.resize(frame_anotado, nuevo_tamano, interpolation=cv2.INTER_AREA)

        if writer is None:
            alto, ancho = frame_anotado.shape[:2]
            writer = cv2.VideoWriter(
                str(ruta_temporal), cv2.VideoWriter_fourcc(*'mp4v'), fps, (ancho, alto)
            )
            if not writer.isOpened():
                cap.release()
                raise RuntimeError(f'No se pudo crear el video de revisión: {ruta_temporal}')

        writer.write(frame_anotado)
        ultimo_frame = frame_anotado
        if len(keypoints) == 30:
            muestra = frame_anotado.copy()

    cap.release()
    if writer is not None:
        writer.release()
        a_h264(ruta_temporal, ruta_overlay)
        if muestra is None:
            muestra = ultimo_frame

    keypoints = np.array(keypoints, dtype=np.float32) if keypoints else np.zeros((0, 17, 2), dtype=np.float32)
    confianzas = np.array(confianzas, dtype=np.float32) if confianzas else np.zeros((0, 17), dtype=np.float32)
    detecciones = np.array(detecciones, dtype=bool)

    return keypoints, confianzas, detecciones, float(fps), muestra, float(fps_original), float(salto_medio)


def estadisticas(clase, nombre, keypoints, confianzas, detectado, fps):
    """Calcula métricas sobre los 8 puntos elegidos y marca los videos que requieren revisión."""
    cantidad_frames = len(keypoints)
    porcentaje_sin_deteccion = float(1 - detectado.mean()) if cantidad_frames else 1.0

    if detectado.any():
        confianza_media = float(confianzas[detectado][:, PUNTOS_TORSO].mean())
    else:
        confianza_media = 0.0

    requiere_revision = (
        cantidad_frames == 0
        or porcentaje_sin_deteccion > UMBRAL_SIN_DETECCION
        or confianza_media < UMBRAL_CONF
    )
    return {
        'clase': clase,
        'video': nombre,
        'frames': cantidad_frames,
        'fps': round(fps, 2),
        'duracion_s': round(cantidad_frames / fps, 2) if fps else 0,
        'pct_sin_deteccion': round(porcentaje_sin_deteccion * 100, 1),
        'conf_media_torso': round(confianza_media, 3),
        'estado': 'REVISAR' if requiere_revision else 'OK',
    }


print('Funciones listas.')

## 5. Encontrar y procesar los videos

El notebook recorre las subcarpetas de `videos/`, procesa los archivos compatibles y guarda los resultados. Si `REPROCESAR = False`, solo reutiliza archivos `.npz` que tengan las dimensiones, metadatos y versión de formato esperados; los archivos antiguos o incompatibles se vuelven a procesar automáticamente.

Cada `.npz` contiene `kpts` (fotogramas × 17 × 2), `conf` (fotogramas × 17), `detectado`, `fps`, `fps_original`, `paso` (salto medio, puede ser decimal), `fps_objetivo` y `version_formato`.

In [ ]:
# Crear una lista de pares (clase, ruta del video).
videos = []
for clase in clases:
    carpeta_clase = VIDEOS / clase
    for ruta in sorted(carpeta_clase.iterdir()):
        if ruta.suffix.lower() in EXTENSIONES and not ruta.name.startswith('.'):
            videos.append((clase, ruta))

if not videos:
    raise RuntimeError(
        f'No encontré videos en {VIDEOS}. Revisa que estén dentro de las subcarpetas de cada clase '
        'y que tengan una extensión compatible.'
    )

print(f'Videos encontrados: {len(videos)}')
filas = []
muestras = {}

for clase, ruta_video in tqdm(videos, desc='Procesando videos'):
    nombre = ruta_video.stem
    (KP_DIR / clase).mkdir(parents=True, exist_ok=True)
    (OV_DIR / clase).mkdir(parents=True, exist_ok=True)
    ruta_keypoints = KP_DIR / clase / f'{nombre}.npz'
    ruta_overlay = OV_DIR / clase / f'{nombre}.mp4'

    reutilizar = False
    datos = None
    if ruta_keypoints.exists() and not REPROCESAR:
        try:
            with np.load(ruta_keypoints, allow_pickle=False) as archivo:
                archivos = set(archivo.files)
                requeridos = {'kpts', 'conf', 'detectado', 'fps', 'fps_objetivo', 'version_formato'}
                if requeridos.issubset(archivos):
                    kpts_temp = archivo['kpts']
                    conf_temp = archivo['conf']
                    detectado_temp = archivo['detectado']
                    fps_temp = float(archivo['fps'])
                    objetivo_temp = float(archivo['fps_objetivo'])
                    version_temp = int(archivo['version_formato'])
                    reutilizar = (
                        kpts_temp.ndim == 3
                        and kpts_temp.shape[1:] == (17, 2)
                        and conf_temp.shape == (len(kpts_temp), 17)
                        and detectado_temp.shape == (len(kpts_temp),)
                        and version_temp == VERSION_FORMATO
                        and objetivo_temp == float(FPS_OBJETIVO or 0)
                        and ruta_overlay.exists()
                    )
                    if reutilizar:
                        keypoints = kpts_temp.copy()
                        confianzas = conf_temp.copy()
                        detectado = detectado_temp.copy()
                        fps = fps_temp
        except Exception as error:
            print(f'  {nombre}: no se pudo reutilizar el archivo guardado ({error}); se reprocesa.')

        if not reutilizar:
            print(f'  {nombre}: archivo ausente, antiguo o incompatible; se reprocesa.')

    if not reutilizar:
        keypoints, confianzas, detectado, fps, muestra, fps_original, paso = procesar_video(
            ruta_video, ruta_overlay
        )
        np.savez_compressed(
            ruta_keypoints,
            kpts=keypoints,
            conf=confianzas,
            detectado=detectado,
            fps=fps,
            fps_original=fps_original,
            paso=paso,
            fps_objetivo=float(FPS_OBJETIVO or 0),
            version_formato=VERSION_FORMATO,
        )
        if muestra is not None and clase not in muestras:
            muestras[clase] = (nombre, muestra)
    else:
        # Si se reutilizan los keypoints, usar el video de revisión existente para la muestra.
        if clase not in muestras and ruta_overlay.exists():
            cap_muestra = cv2.VideoCapture(str(ruta_overlay))
            cap_muestra.set(cv2.CAP_PROP_POS_FRAMES, 29)
            ok, imagen_muestra = cap_muestra.read()
            if not ok:
                cap_muestra.set(cv2.CAP_PROP_POS_FRAMES, 0)
                ok, imagen_muestra = cap_muestra.read()
            cap_muestra.release()
            if ok:
                muestras[clase] = (nombre, imagen_muestra)

    filas.append(estadisticas(clase, nombre, keypoints, confianzas, detectado, fps))

# Crear el resumen general y exportarlo a CSV.
df = pd.DataFrame(filas)
ruta_manifest = SALIDA / 'manifest.csv'
df.to_csv(ruta_manifest, index=False, encoding='utf-8')
print('Procesamiento terminado. Resumen guardado en:', ruta_manifest)
display(df)

## 6. Generar gráficos de evidencia

Se generan gráficos para comparar la cantidad de videos y fotogramas por clase, además de una imagen de ejemplo con el esqueleto detectado.

In [ ]:
# Resumir la cantidad de videos y frames de cada clase.
resumen = (
    df.groupby('clase')
    .agg(videos=('video', 'count'), frames=('frames', 'sum'))
    .reset_index()
)

fig, ejes = plt.subplots(1, 2, figsize=(10, 4))

ejes[0].bar(resumen['clase'], resumen['videos'])
ejes[0].set_title('Videos por clase')
ejes[0].set_xlabel('Clase')
ejes[0].set_ylabel('Cantidad de videos')

ejes[1].bar(resumen['clase'], resumen['frames'])
ejes[1].set_title('Frames por clase')
ejes[1].set_xlabel('Clase')
ejes[1].set_ylabel('Cantidad de frames')

plt.tight_layout()
plt.savefig(SALIDA / 'balance_clases.png', dpi=150)
plt.show()

# Mostrar una imagen de ejemplo por cada clase procesada en esta ejecución.
if muestras:
    fig, ejes = plt.subplots(
        1,
        len(muestras),
        figsize=(4 * len(muestras), 4),
        squeeze=False,
    )

    for eje, (clase, (nombre, imagen)) in zip(ejes[0], muestras.items()):
        imagen_rgb = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
        eje.imshow(imagen_rgb)
        eje.set_title(f'{clase}\n{nombre}', fontsize=9)
        eje.axis('off')

    plt.tight_layout()
    plt.savefig(SALIDA / 'muestras_esqueleto.png', dpi=150)
    plt.show()
else:
    print(
        'No hay muestras nuevas porque los videos ya estaban procesados. '
        'Cambia REPROCESAR = True en la sección 2 si quieres generarlas de nuevo.'
    )

print('Revisa los resultados en:', SALIDA)

## 7. Cómo interpretar los resultados

- **`OK`**: el video cumple los umbrales definidos para detección y confianza, calculada sobre los 8 keypoints elegidos.
- **`REVISAR`**: hay demasiados fotogramas sin detección, la confianza media es baja o el video no contiene fotogramas procesables.

Contenido de cada archivo `salida/keypoints/<clase>/<video>.npz`:

- **`kpts`**: coordenadas `(x, y)` en píxeles de los 17 keypoints COCO, con forma `(frames, 17, 2)`.
- **`conf`**: confianza de cada keypoint, con forma `(frames, 17)`.
- **`detectado`**: indica si se encontró una persona en cada fotograma procesado.
- **`fps`**: frecuencia temporal efectiva del resultado.
- **`fps_original`**: FPS informado por el video original.
- **`paso`**: salto medio entre fotogramas originales; puede ser decimal para aproximarse mejor al FPS objetivo.
- **`fps_objetivo`** y **`version_formato`**: parámetros usados para saber si el archivo se puede reutilizar.

### Keypoints que usará inicialmente el LSTM

Aunque se guardan los 17 puntos, inicialmente seleccionaremos 8 mediante `PUNTOS_TORSO`:

| Índice COCO | Keypoint |
|---|---|
| 5, 6 | hombro izquierdo y derecho |
| 7, 8 | codo izquierdo y derecho |
| 9, 10 | muñeca izquierda y derecha |
| 11, 12 | cadera izquierda y derecha |

El video de revisión dibuja solamente estos ocho puntos y sus conexiones, para que la nariz, los ojos y las piernas no distraigan la inspección visual. Los 17 puntos quedan guardados por si más adelante queremos evaluar otra selección.

Los umbrales son configurables. Un estado `OK` no garantiza que cada keypoint sea perfecto; revisa también los videos de salida y los casos marcados como `REVISAR`.